# Notebook 3 — Chuẩn hóa văn bản phương ngữ bằng mBART và LoRA

Notebook này là phần chính của đồ án: dùng một mô hình **encoder-decoder**
để dịch câu phương ngữ về câu tiếng Việt chuẩn, fine-tune cùng checkpoint bằng
LoRA, rồi kiểm tra normalization có phục hồi downstream task performance không.

## Mục tiêu học tập

1. Giải thích vì sao bài toán normalization dùng mô hình seq2seq, không phải encoder-only.
2. Chia train/dev theo `sample_id`, không dùng test để chọn thiết kế.
3. Đo CER/WER/chrF và NLL familiarity gap; dùng CER làm primary metric.
4. Fine-tune **cùng private mBART checkpoint** bằng LoRA và chọn best checkpoint
   theo dev CER.
5. Đánh giá baseline và LoRA đúng một lần trên cùng test split.
6. Đo downstream recovery trên một task phân loại và phân tích lỗi thủ công.

## Khái niệm: text normalization là gì?

**Text normalization** là phép biến đổi một văn bản về một dạng chuẩn, giữ nguyên
nghĩa. Trong đồ án này, "dạng chuẩn" = tiếng Việt phổ thông, và input = câu viết
bằng phương ngữ (PNB/PNT2/PNT3).

Đây là bài toán **sequence-to-sequence (seq2seq)**: đọc một chuỗi dialect, sinh
một chuỗi standard. Phép biến đổi có thể là thay từ vựng, sửa hậu tố, đổi ngữ pháp
hoặc để nguyên khi câu đã chuẩn. Vì output là chuỗi tự do (không phải nhãn), ta
cần một mô hình có cả encoder (đọc input) và decoder (sinh output).

Tham khảo CS224n: mô hình encoder-decoder factorize `p(y|x)` thành
`encoder(x) → context → decoder(context, y_<t) → y_t`. Đây chính là kiến trúc
của mBART.

## mBERT hay mBART? Phân biệt quan trọng

Đồ án dùng **mBART** (`facebook/mbart-large-50` / checkpoint private
`tarudesu/mbart-large-50`), **không phải mBERT**. Hai mô hình này rất khác nhau:

| Khía cạnh | mBERT | mBART |
|---|---|---|
| Kiến trúc | encoder-only | encoder-decoder (seq2seq) |
| Hàm loss | masked LM | denoising seq2seq |
| Output | biểu diễn ẩn / nhãn | chuỗi token tự do |
| Phù hợp | phân loại, embedding | dịch, tóm tắt, **normalization** |

Vì bài toán normalization cần **sinh chuỗi**, mBERT (encoder-only) không làm được
trực tiếp. mBART có decoder nên gán được `p(standard | dialect)`. Repo chưa công
bố bảng so sánh đủ để tuyên bố checkpoint private là tốt nhất; kết quả rehearsal
phải được ghi vào `data/model_results/normalization_model_comparison.csv`.

> **Lưu ý giảng dạy:** Tên hai mô hình dễ nhầm. Nếu đọc "mBERT" trong tài liệu cũ,
> hãy kiểm tra kiến trúc: nếu nó dịch được chuỗi, đó là mBART.

In [ ]:
from pathlib import Path
import sys

# Detect project root whether we run from notebooks/ or project root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data" / "train_240.jsonl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
print(f"Project root: {ROOT}")

In [ ]:
import subprocess
import sys

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5", "torch>=2.2,<3",
        "sentencepiece>=0.2,<1", "peft>=0.11,<1",
        "datasets>=2.19,<3", "accelerate>=0.30,<1",
        "evaluate>=0.4,<1", "sacrebleu>=2.4,<3",
    ])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vialect_seas.data import (
    DIALECTS, TASKS, load_jsonl, split_train_dev_by_source,
)
from vialect_seas.metrics import (
    evaluate_predictions, identity_metric_summary, metric_summary,
    paired_cluster_bootstrap,
)
from vialect_seas.normalization import (
    EXPERIMENT_START_MODEL_ID, get_hf_token, load_experiment_start_model,
    generate_normalizations, attach_lora, make_preprocess_function,
    resolved_model_revision, save_experiment_config, load_experiment_config,
)

sns.set_theme(style="whitegrid", context="notebook")
train_pool = load_jsonl(ROOT / "data" / "train_240.jsonl")
test_path = ROOT / "data" / "test_300.jsonl"
test_row_count = sum(1 for line in test_path.open(encoding="utf-8") if line.strip())
train, dev = split_train_dev_by_source(
    train_pool, dev_sources_per_task=4, seed=2026
)
print(f"train={len(train)}  dev={len(dev)}  held-out test rows={test_row_count}")
print("Source overlap:", {
    "train-dev": len(set(train.sample_id) & set(dev.sample_id)),
})
display(train.groupby(["task", "target_dialect"], observed=True).size().unstack())
assert len(train) == 192 and len(dev) == 48 and test_row_count == 300

## Protocol cố định

| Split | Source/task | Rows | Vai trò |
|---|---:|---:|---|
| Train | 16 | 192 | cập nhật LoRA |
| Dev | 4 | 48 | chọn rank, epoch, checkpoint và decoding |
| Test | 25 | 300 | đánh giá cuối sau khi khóa thiết kế |

Baseline và LoRA cùng bắt đầu từ `EXPERIMENT_START_MODEL_ID`. Không dùng test để
quyết định có fine-tune, chọn hyperparameter hoặc chọn checkpoint.

### STUDENT TASK 0 — Kế hoạch thí nghiệm

Trước khi tải mô hình, ghi lại dự đoán của nhóm. Đây là preregistration: giúp
nhóm không tự điều chỉnh hypothesis sau khi thấy kết quả.

In [ ]:
# HINT: RQ nên nêu rõ (1) input, (2) yếu tố so sánh, (3) metric.
"""Your code here"""
TEAM_NAME = "TODO"
NORM_RQ = "TODO: LoRA adaptation có giảm dev CER và phục hồi task accuracy không?"
PRIMARY_METRIC = "dev CER"
PRED_CER = "TODO: LoRA CER so với baseline trên dev"
PRED_NLL_RECOVERY = "TODO: dương / xấp xỉ 0 / âm"
SUCCESS_CRITERION = "TODO: ngưỡng giảm dev CER định lượng trước training"
DOWNSTREAM_TASK = "SENT"

print({
    "team": TEAM_NAME,
    "rq": NORM_RQ,
    "primary_metric": PRIMARY_METRIC,
    "pred_cer": PRED_CER,
    "pred_nll_recovery": PRED_NLL_RECOVERY,
    "success_criterion": SUCCESS_CRITERION,
    "downstream_task": DOWNSTREAM_TASK,
})

## Baseline trên development split

Experiment checkpoint là `EXPERIMENT_START_MODEL_ID` (private mBART). Token
**không được hardcode**
trong notebook. Đặt `HF_TOKEN` trong biến môi trường hoặc Colab Secrets:

```python
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PRIVATE_NORMALIZER_REVISION"] = userdata.get(
    "PRIVATE_NORMALIZER_REVISION"
)
```

Hàm `get_hf_token()` đọc token từ env hoặc Colab Secrets. **Không bao giờ in
token ra output, không commit file `.env`.**

Mỗi học viên dùng token **read-only của chính mình** sau khi được cấp quyền vào
model repo. Không chia sẻ personal token chung trong lớp. TA có thể cấp quyền
theo từng tài khoản hoặc dùng organization/gated model; nếu không cấp quyền,
TA chạy trước và phát output baseline đã lưu.

In [ ]:
# Tự nạp Colab Secrets nếu notebook đang chạy trên Colab.
import os

loaded_secrets = []
try:
    from google.colab import userdata
except ImportError:
    print("Không chạy trên Colab; dùng biến môi trường hiện có.")
else:
    for secret_name in ("HF_TOKEN", "PRIVATE_NORMALIZER_REVISION"):
        if os.environ.get(secret_name):
            continue
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
            loaded_secrets.append(secret_name)
    print("Đã nạp Colab Secrets:", loaded_secrets or "không có secret mới")

In [ ]:
# Kiểm tra token có sẵn không (không in giá trị token).
token = get_hf_token(required=False)
private_revision = os.environ.get("PRIVATE_NORMALIZER_REVISION")
print("HF_TOKEN available:", token is not None)
print(
    "Private revision configured:",
    private_revision is not None and len(private_revision) == 40,
)

In [ ]:
RUN_DEV_BASELINE = False  # Đổi True khi đã có GPU và token.
dev_frame = dev.sort_values(
    ["task", "target_dialect", "sample_id"]
).reset_index(drop=True)
print("Development rows:", len(dev_frame))
display(dev_frame[[
    "sample_id", "task", "target_dialect", "dialect_text", "standard_text"
]].head())

In [ ]:
output_path = ROOT / "outputs" / "dev_baseline_predictions.csv"
output_path.parent.mkdir(exist_ok=True)

if RUN_DEV_BASELINE:
    tokenizer, model, device = load_experiment_start_model()
    print(f"Loaded {EXPERIMENT_START_MODEL_ID} on {device}")
    normalized = generate_normalizations(
        dev_frame["dialect_text"].tolist(),
        tokenizer, model, device,
        max_length=192, batch_size=8,
    )
    dev_baseline = dev_frame.copy()
    dev_baseline["prediction"] = normalized
    dev_baseline = evaluate_predictions(dev_baseline)
    dev_baseline.to_csv(output_path, index=False)
    print("Saved", output_path)
    # Giải phóng VRAM trước khi tải LM ở dưới.
    import gc, torch
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif output_path.exists():
    dev_baseline = pd.read_csv(output_path)
    print("Loaded existing", output_path)
else:
    dev_baseline = pd.DataFrame()
    print("Set RUN_DEV_BASELINE=True (cần GPU + HF_TOKEN) để tạo dev baseline")

## Diagnostic: độ dài và NLL familiarity

Đây là phần quan trọng nhất của notebook. Với mỗi cặp, ta có **ba phiên bản** của
cùng một ý nghĩa:

1. **`dialect_text`** — câu gốc (input phương ngữ đưa vào normalizer).
2. **`prediction`** — câu chuẩn hóa (output của mBART).
3. **`standard_text`** — bản chuẩn vàng (reference).

Ta so sánh ba phiên bản bằng độ dài và token NLL từ một reference LM:

- **Độ dài** (số từ): bản dịch có dài/ngắn bất thường so với gốc và chuẩn không?
- **NLL familiarity:** NLL thấp hơn nghĩa là reference scorer thấy chuỗi quen hơn.
  Nó không đo chất lượng, độ khó ngôn ngữ, ngữ pháp hay bảo toàn nghĩa.

```text
NLL(text) = -(1/(T-1)) * Σ log p(x_t | x_<t)
g_dialect = NLL(dialect) - NLL(standard)
g_normalized = NLL(normalized) - NLL(standard)
NLLRecovery = g_dialect - g_normalized
```

`NLLRecovery > 0` nghĩa là output đã tiến gần hơn tới phân phối của standard dưới
scorer này. Đây chỉ là diagnostic; chọn model vẫn dựa trên **dev CER**.

In [ ]:
RUN_NLL_DIAGNOSTIC = False  # Đổi True khi đã có dev baseline + GPU.

if not dev_baseline.empty:
    # --- Độ dài: số từ của ba phiên bản ---
    length_frame = dev_baseline.assign(
        dialect_words=dev_baseline["dialect_text"].str.split().str.len(),
        normalized_words=dev_baseline["prediction"].str.split().str.len(),
        standard_words=dev_baseline["standard_text"].str.split().str.len(),
    )
    length_long = length_frame.melt(
        id_vars=["sample_id", "task", "target_dialect"],
        value_vars=["dialect_words", "normalized_words", "standard_words"],
        var_name="variant", value_name="n_words",
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    sns.boxplot(data=length_long, x="target_dialect", y="n_words", hue="variant", ax=ax)
    ax.set(title="Độ dài (số từ): câu gốc vs câu dịch vs chuẩn vàng",
           xlabel="Dialect", ylabel="Số từ")
    plt.tight_layout()
    plt.show()

    length_summary = (
        length_long.groupby(["target_dialect", "variant"], observed=True)
        .n_words.mean().unstack()
        .rename(columns={
            "dialect_words": "dialect(gốc)",
            "normalized_words": "normalized(dịch)",
            "standard_words": "standard(vàng)",
        })
    )
    display(length_summary.round(2))

### Insight (độ dài)

Viết 2–3 câu theo cấu trúc observation → evidence → caveat:

- Bản dịch có dài/ngắn hơn câu gốc không? Có tiệm cận chuẩn vàng không?
- Có khác biệt giữa ba dialect không?
- **Caveat:** độ dài gần nhau không chứng minh nghĩa đã được chuẩn hóa đúng.

In [ ]:
# --- NLL familiarity của ba phiên bản bằng LM tham chiếu ---
from vialect_seas.probing import load_causal_lm, score_texts

REFERENCE_LM = "Qwen/Qwen2.5-0.5B"
nll_path = ROOT / "outputs" / "dev_normalization_nll.csv"

if RUN_NLL_DIAGNOSTIC and not dev_baseline.empty:
    tok, lm, lm_device = load_causal_lm(REFERENCE_LM)
    print(f"Loaded {REFERENCE_LM} on {lm_device}")

    scored_dialect = score_texts(dev_baseline["dialect_text"], tok, lm, lm_device)
    scored_normalized = score_texts(dev_baseline["prediction"], tok, lm, lm_device)
    scored_standard = score_texts(dev_baseline["standard_text"], tok, lm, lm_device)

    nll_frame = dev_baseline[["sample_id", "task", "target_dialect"]].copy()
    nll_frame["nll_dialect"] = scored_dialect["nll"].values
    nll_frame["nll_normalized"] = scored_normalized["nll"].values
    nll_frame["nll_standard"] = scored_standard["nll"].values
    nll_frame["g_dialect"] = nll_frame["nll_dialect"] - nll_frame["nll_standard"]
    nll_frame["g_normalized"] = nll_frame["nll_normalized"] - nll_frame["nll_standard"]
    nll_frame["nll_recovery"] = nll_frame["g_dialect"] - nll_frame["g_normalized"]
    nll_frame.to_csv(nll_path, index=False)
    print("Saved", nll_path)

    import gc, torch
    del lm, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif nll_path.exists():
    nll_frame = pd.read_csv(nll_path)
    print("Loaded existing", nll_path)
else:
    nll_frame = pd.DataFrame()
    print("Set RUN_NLL_DIAGNOSTIC=True để tính NLL diagnostic trên dev")

In [ ]:
if not nll_frame.empty:
    gap_long = nll_frame.melt(
        id_vars=["sample_id", "task", "target_dialect"],
        value_vars=["g_dialect", "g_normalized"],
        var_name="comparison", value_name="nll_gap",
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.boxplot(data=gap_long, x="target_dialect", y="nll_gap",
                hue="comparison", ax=axes[0])
    axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0].set(
        title="NLL gap relative to Standard",
        xlabel="Dialect",
        ylabel="NLL(input) - NLL(Standard)",
    )
    sns.barplot(
        data=nll_frame, x="target_dialect", y="nll_recovery", ax=axes[1]
    )
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set(
        title="NLL recovery after normalization",
        xlabel="Dialect",
        ylabel="g_dialect - g_normalized",
    )
    plt.tight_layout()
    plt.show()

    nll_summary = nll_frame.groupby("target_dialect", observed=True).agg(
        g_dialect=("g_dialect", "mean"),
        g_normalized=("g_normalized", "mean"),
        nll_recovery=("nll_recovery", "mean"),
    )
    display(nll_summary.round(3))

### Insight (NLL diagnostic)

Trả lời các câu sau bằng số từ biểu đồ:

1. `g_dialect` có dương không và dialect nào lớn nhất?
2. `g_normalized` có nhỏ hơn `g_dialect` không?
3. `NLLRecovery` có dương nhất quán theo dialect không?

**Caveat:** NLL chỉ đo familiarity dưới reference scorer. Không dùng NLL recovery
để chọn checkpoint và không gọi chuỗi có NLL thấp hơn là “tốt hơn”.

## Đánh giá bản dịch bằng metric dịch máy

Primary metric là CER trên dev. Báo cáo thêm:

- **CER (Character Error Rate)** = edit distance / ký tự reference.
- **WER (Word Error Rate)** = edit distance / từ reference.
- **chrF** = character n-gram F-score; cao hơn tốt hơn.
- **Exact match** = 1 nếu bản dịch == chuẩn vàng (sau khi strip).

CER thấp = bản dịch giống chuẩn vàng về bề mặt. Nhưng CER không thưởng nghĩa;
dùng kèm kiểm tra thủ công.

In [ ]:
if not dev_baseline.empty:
    dev_baseline = evaluate_predictions(dev_baseline)
    metric_by_dialect = metric_summary(
        dev_baseline, by=["target_dialect"]
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=metric_by_dialect, x="target_dialect", y="cer", ax=ax)
    ax.set(title="Development CER của mBART baseline",
           xlabel="Dialect", ylabel="Character Error Rate")
    plt.tight_layout()
    plt.show()
    display(metric_by_dialect.round(3))
    display(metric_summary(dev_baseline).round(3))

### STUDENT TASK 1 — Phân tích lỗi thủ công

Metric chỉ cho số. Nhóm phải đọc bản dịch và phân loại lỗi. Chọn ít nhất 10 câu
có CER cao nhất và gán nhãn lỗi theo taxonomy: `lexical` (sai từ vựng),
`morphology` (hậu tố/chia thì), `syntax` (trật tự từ), `discourse` (mạch lạc),
`meaning` (sai nghĩa), `fluent` (không lỗi).

In [ ]:
RUN_ERROR_ANALYSIS = False

if RUN_ERROR_ANALYSIS and not dev_baseline.empty:
    # HINT 1: sort theo CER giảm dần, lấy top 10-15 câu.
    # HINT 2: với mỗi câu, in dialect_text / prediction / standard_text cạnh nhau.
    # HINT 3: gán nhãn lỗi vào cột `error_type`; đếm tần suất theo error_type.
    """Your code here"""
    top_errors = None  # thay bằng DataFrame của nhóm

    # SELF-CHECK
    assert top_errors is not None, "Hãy tạo biến top_errors"
    assert "error_type" in top_errors.columns
    assert len(top_errors) >= 10
    display(top_errors[["sample_id", "target_dialect", "dialect_text",
                        "prediction", "standard_text", "cer", "error_type"]].head(12))

## Fine-tune bằng LoRA

Ta kiểm tra giả thuyết LoRA đã đăng ký bằng train/dev, không dùng NLL/PPL để
quyết định sau khi nhìn test. mBART-large có hàng trăm triệu tham số; LoRA
(Low-Rank Adaptation) chỉ cập nhật các ma trận thấp hạng.

```text
W' = W + (alpha / r) * B @ A
```

- `W` ∈ R^(d×d): weight gốc, đóng băng.
- `A` ∈ R^(r×d), `B` ∈ R^(d×r): hai ma trận thấp hạng, rank `r` nhỏ (ví dụ 8).
- `alpha`: scaling.
- Chỉ `A`, `B` được update; tham số gốc giữ nguyên → tiết kiệm VRAM.

Tham khảo CS224n: đây là một dạng **parameter-efficient fine-tuning**. Trong đồ
án, LoRA gắn vào `q_proj` và `v_proj` của attention (các projection hay nhạy với
tác vụ).

### Cấu hình và model selection

Điền cột **Giải thích ý nghĩa** trước khi train. Mọi thay đổi phải chọn bằng dev;
test vẫn đóng.

| Argument | Giá trị Gợi ý | Giải thích ý nghĩa |
| --- | --- | --- |
| `output_dir` | `"outputs/lora_run"` | VD: đường dẫn thư mục lưu kết quả training |
| `num_train_epochs` | `3` | |
| `per_device_train_batch_size` | `4` | |
| `per_device_eval_batch_size` | `4` | |
| `learning_rate` | `2e-4` | |
| `warmup_ratio` | `0.05` | |
| `weight_decay` | `0.01` | |
| `eval_strategy` | `"epoch"` | đánh giá trên dev mỗi epoch |
| `save_strategy` | `"epoch"` | lưu ứng viên checkpoint |
| `logging_steps` | `10` | |
| `report_to` | `[]` | |
| `fp16` | **True** (nếu có GPU) | |
| `predict_with_generate` | **True** | sinh chuỗi để tính dev CER |
| `metric_for_best_model` | `"cer"` | primary selection metric |
| `greater_is_better` | `False` | CER thấp hơn tốt hơn |
| `gradient_accumulation_steps` | `4` | |

In [ ]:
RUN_FINETUNE = False
EXPERIMENT_CONFIG = {
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "epochs": 3,
    "train_batch_size": 4,
    "eval_batch_size": 4,
    "gradient_accumulation": 4,
    "learning_rate": 2e-4,
    "max_length": 192,
}
experiment_config_path = ROOT / "outputs" / "experiment_config.json"
# STUDENT TASK: giải thích trade-off của từng giá trị trước khi bật training.
# HINT: nêu giới hạn dữ liệu/VRAM và giả thuyết overfitting.
HYPERPARAMETER_RATIONALE = {
    key: "TODO" for key in EXPERIMENT_CONFIG
}
"""Your code here"""

if RUN_FINETUNE:
    import torch
    from datasets import Dataset
    from transformers import (
        DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
    )

    unresolved = [
        key for key, value in HYPERPARAMETER_RATIONALE.items()
        if value == "TODO"
    ]
    if unresolved:
        raise ValueError(f"Giải thích hyperparameter trước khi train: {unresolved}")

    # Controlled comparison: exact same checkpoint as baseline.
    tokenizer, base_model, device = load_experiment_start_model()
    model_revision = resolved_model_revision(base_model)
    model = attach_lora(
        base_model,
        rank=EXPERIMENT_CONFIG["lora_rank"],
        alpha=EXPERIMENT_CONFIG["lora_alpha"],
        dropout=EXPERIMENT_CONFIG["lora_dropout"],
    )
    model.print_trainable_parameters()

    preprocess = make_preprocess_function(
        tokenizer, max_length=EXPERIMENT_CONFIG["max_length"]
    )
    train_ds = Dataset.from_pandas(
        train[["dialect_text", "standard_text"]], preserve_index=False
    )
    dev_ds = Dataset.from_pandas(
        dev[["dialect_text", "standard_text"]], preserve_index=False
    )
    train_tok = train_ds.map(
        preprocess, batched=True, remove_columns=train_ds.column_names
    )
    dev_tok = dev_ds.map(
        preprocess, batched=True, remove_columns=dev_ds.column_names
    )
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model, label_pad_token_id=-100
    )

    def compute_generation_metrics(eval_pred):
        predictions, labels = eval_pred
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_predictions = tokenizer.batch_decode(
            predictions, skip_special_tokens=True
        )
        decoded_labels = tokenizer.batch_decode(
            labels, skip_special_tokens=True
        )
        scored = pd.DataFrame({
            "standard_text": decoded_labels,
            "prediction": decoded_predictions,
        })
        return metric_summary(evaluate_predictions(scored)).iloc[0].to_dict()

    args = Seq2SeqTrainingArguments(
        output_dir=str(ROOT / "outputs" / "lora_run"),
        num_train_epochs=EXPERIMENT_CONFIG["epochs"],
        per_device_train_batch_size=EXPERIMENT_CONFIG["train_batch_size"],
        per_device_eval_batch_size=EXPERIMENT_CONFIG["eval_batch_size"],
        gradient_accumulation_steps=EXPERIMENT_CONFIG["gradient_accumulation"],
        learning_rate=EXPERIMENT_CONFIG["learning_rate"],
        warmup_ratio=0.05,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="cer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=EXPERIMENT_CONFIG["max_length"],
        save_total_limit=1,
        logging_steps=10,
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=2026,
        data_seed=2026,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_generation_metrics,
    )
    trainer.train()
    trainer.save_model(str(ROOT / "outputs" / "lora_adapter"))
    tokenizer.save_pretrained(str(ROOT / "outputs" / "lora_adapter"))
    locked_manifest = save_experiment_config(
        EXPERIMENT_CONFIG,
        experiment_config_path,
        model_revision=model_revision,
        extra={
            "seed": 2026,
            "train_rows": len(train),
            "dev_rows": len(dev),
            "selection_metric": "cer",
            "best_checkpoint": trainer.state.best_model_checkpoint,
        },
    )
    print("Saved LoRA adapter to outputs/lora_adapter")
    print("Config SHA-256:", locked_manifest["config_sha256"])
else:
    print("Set RUN_FINETUNE=True after completing the experiment plan.")

## Final test — chỉ mở sau khi khóa thiết kế

Đây là lần đầu test 300 được dùng để sinh prediction. Baseline và LoRA:

- bắt đầu từ cùng `EXPERIMENT_START_MODEL_ID`;
- dùng cùng 300 dòng, thứ tự và decoding;
- khác nhau duy nhất ở LoRA adapter;
- được báo cáo bằng CER/WER/chrF/exact match.

In [ ]:
RUN_FINAL_TEST = False
CONFIG_LOCKED = False  # Chỉ đổi True sau khi chọn checkpoint bằng dev CER.
adapter_path = ROOT / "outputs" / "lora_adapter"
final_path = ROOT / "outputs" / "test_baseline_vs_lora.csv"
locked_config = None

if CONFIG_LOCKED:
    assert experiment_config_path.exists(), (
        "Chưa có outputs/experiment_config.json từ training run"
    )
    locked_config = load_experiment_config(experiment_config_path)
    assert locked_config["experiment_config"] == EXPERIMENT_CONFIG, (
        "EXPERIMENT_CONFIG đã đổi sau khi lưu manifest"
    )

if RUN_FINAL_TEST:
    assert CONFIG_LOCKED, "Khóa hyperparameter/checkpoint bằng dev trước khi mở test"
    assert adapter_path.exists(), "Chưa có LoRA adapter"
    import gc
    import torch
    from peft import PeftModel

    test = load_jsonl(test_path)
    locked_revision = locked_config["model_revision"]
    baseline_tokenizer, baseline_model, baseline_device = (
        load_experiment_start_model(revision=locked_revision)
    )
    normalized_baseline = generate_normalizations(
        test["dialect_text"].tolist(),
        baseline_tokenizer, baseline_model, baseline_device,
        max_length=EXPERIMENT_CONFIG["max_length"], batch_size=4,
    )
    del baseline_model, baseline_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    lora_tokenizer, lora_base, lora_device = (
        load_experiment_start_model(revision=locked_revision)
    )
    lora_model = PeftModel.from_pretrained(lora_base, adapter_path).to(lora_device)
    lora_model.eval()
    normalized_lora = generate_normalizations(
        test["dialect_text"].tolist(),
        lora_tokenizer, lora_model, lora_device,
        max_length=EXPERIMENT_CONFIG["max_length"], batch_size=4,
    )

    comparison = test.copy()
    comparison["normalized_baseline"] = normalized_baseline
    comparison["normalized_lora"] = normalized_lora
    for variant, column in {
        "baseline": "normalized_baseline",
        "lora": "normalized_lora",
    }.items():
        scored = evaluate_predictions(
            comparison, prediction_column=column
        )
        for metric in ["cer", "wer", "chrf", "exact_match", "length_ratio"]:
            comparison[f"{variant}_{metric}"] = scored[metric]
    comparison.to_csv(final_path, index=False)
    print("Saved", final_path)
elif final_path.exists():
    comparison = pd.read_csv(final_path)
    print("Loaded existing locked-test results", final_path)
else:
    comparison = pd.DataFrame()
    print("Test remains closed. Lock config before RUN_FINAL_TEST=True.")

if not comparison.empty:
    required = {
        "normalized_baseline", "normalized_lora",
        "baseline_cer", "lora_cer", "baseline_chrf", "lora_chrf",
    }
    assert required.issubset(comparison.columns)
    assert len(comparison) == 300
    summary_rows = []
    for variant in ["baseline", "lora"]:
        for dialect, group in comparison.groupby("target_dialect", observed=True):
            summary_rows.append({
                "variant": variant,
                "target_dialect": dialect,
                "cer": group[f"{variant}_cer"].mean(),
                "wer": group[f"{variant}_wer"].mean(),
                "chrf": group[f"{variant}_chrf"].mean(),
                "exact_match": group[f"{variant}_exact_match"].mean(),
            })
    final_summary = pd.DataFrame(summary_rows)
    display(final_summary.round(4))

    cer_long = comparison[[
        "sample_id", "target_dialect", "baseline_cer", "lora_cer",
    ]].melt(
        id_vars=["sample_id", "target_dialect"],
        value_vars=["baseline_cer", "lora_cer"],
        var_name="variant",
        value_name="cer",
    )
    cer_long["variant"] = cer_long["variant"].map({
        "baseline_cer": "baseline",
        "lora_cer": "lora",
    })
    overall_cer_ci = paired_cluster_bootstrap(
        cer_long.assign(scope="overall"),
        value_column="cer",
        group_by=["scope"],
        baseline_variant="baseline",
        comparison_variant="lora",
        n_resamples=5000,
        confidence_level=0.95,
        seed=2026,
    )
    dialect_cer_ci = paired_cluster_bootstrap(
        cer_long,
        value_column="cer",
        group_by=["target_dialect"],
        baseline_variant="baseline",
        comparison_variant="lora",
        n_resamples=5000,
        confidence_level=0.95,
        seed=2026,
    )
    dialect_cer_ci["scope"] = (
        "dialect:" + dialect_cer_ci["target_dialect"].astype(str)
    )
    cer_improvement_ci = pd.concat(
        [
            overall_cer_ci,
            dialect_cer_ci.drop(columns=["target_dialect"]),
        ],
        ignore_index=True,
    )
    cer_improvement_ci.insert(
        1, "effect", "CER_baseline - CER_LoRA"
    )
    cer_improvement_ci.insert(2, "confidence_level", 0.95)
    display(cer_improvement_ci[[
        "scope", "effect", "confidence_level", "mean", "ci_low", "ci_high",
        "n_sources", "n_resamples",
    ]].round(4))

    identity_rows = []
    for variant, column in {
        "baseline": "normalized_baseline",
        "lora": "normalized_lora",
    }.items():
        scored = evaluate_predictions(
            comparison, prediction_column=column
        )
        identity_summary = identity_metric_summary(scored)
        identity_summary.insert(0, "variant", variant)
        identity_rows.append(identity_summary)
    final_identity_summary = pd.concat(identity_rows, ignore_index=True)
    display(final_identity_summary.round(4))

### Insight (final normalization)

Trả lời:

1. LoRA giảm test CER bao nhiêu so với đúng starting checkpoint?
2. Bootstrap CI 95% của `CER_baseline - CER_LoRA` có chứa 0 không?
3. Kết quả có nhất quán trên PNB/PNT2/PNT3 và chrF/WER không?
4. Dev improvement có chuyển sang test không?
5. CER khác nhau thế nào giữa identity và non-identity pairs?
6. Caveat: 192 training rows, target noise và identity pairs.

## Diagnostic cuối: LoRA có giảm NLL gap không?

Phần này báo `NLLRecovery` cho baseline và LoRA trên cùng test rows. Đây là
diagnostic phụ, không thay đổi model đã chọn.

In [ ]:
RUN_FINAL_NLL_DIAGNOSTIC = False
final_nll_path = ROOT / "outputs" / "test_nll_recovery.csv"

if RUN_FINAL_NLL_DIAGNOSTIC and not comparison.empty:
    tok, lm, lm_device = load_causal_lm(REFERENCE_LM)
    variant_columns = {
        "standard": "standard_text",
        "dialect": "dialect_text",
        "baseline": "normalized_baseline",
        "lora": "normalized_lora",
    }
    scored_variants = {
        name: score_texts(comparison[column], tok, lm, lm_device)["nll"].values
        for name, column in variant_columns.items()
    }
    final_nll = comparison[["sample_id", "task", "target_dialect"]].copy()
    for name, values in scored_variants.items():
        final_nll[f"nll_{name}"] = values
    final_nll["baseline_nll_recovery"] = (
        final_nll["nll_dialect"] - final_nll["nll_baseline"]
    )
    final_nll["lora_nll_recovery"] = (
        final_nll["nll_dialect"] - final_nll["nll_lora"]
    )
    final_nll.to_csv(final_nll_path, index=False)
    display(final_nll.groupby("target_dialect")[[
        "baseline_nll_recovery", "lora_nll_recovery"
    ]].mean().round(4))

## Bắt buộc: downstream task recovery

CER tốt hơn chưa chứng minh model downstream robust hơn. Dùng một LM cố định
chấm cùng sample ở bốn input: Standard, Dialect, baseline-normalized và
LoRA-normalized. Primary table báo accuracy và:

```text
Recovery_baseline = Accuracy(normalized_baseline) - Accuracy(dialect)
Recovery_LoRA = Accuracy(normalized_lora) - Accuracy(dialect)
```

In [ ]:
RUN_DOWNSTREAM_RECOVERY = False
downstream_path = ROOT / "outputs" / "test_downstream_recovery.csv"

if RUN_DOWNSTREAM_RECOVERY:
    assert not comparison.empty, "Chạy locked final test trước"
    from vialect_seas.probing import (
        load_text_generator, probe_classification_rows,
    )

    downstream_frame = comparison[
        comparison["task"].eq(DOWNSTREAM_TASK)
    ].copy()
    runner = load_text_generator("Qwen/Qwen2.5-0.5B")
    downstream_scores = probe_classification_rows(
        downstream_frame,
        runner,
        variants=(
            "standard", "dialect",
            "normalized_baseline", "normalized_lora",
        ),
    )
    downstream_scores.to_csv(downstream_path, index=False)
    downstream_accuracy = (
        downstream_scores.groupby("variant", observed=True)
        .correct.mean()
    )
    recovery_table = pd.DataFrame({
        "accuracy": downstream_accuracy,
    })
    dialect_accuracy = downstream_accuracy["dialect"]
    recovery_table["recovery_vs_dialect"] = (
        recovery_table["accuracy"] - dialect_accuracy
    )
    display(recovery_table.round(4))

    # SELF-CHECK
    assert {
        "standard", "dialect", "normalized_baseline", "normalized_lora"
    } == set(recovery_table.index)
    assert downstream_scores["sample_id"].nunique() > 0

## IV. Demo

Chọn một dòng từ `comparison`, trình bày dialect/baseline/LoRA/reference và giải
thích bằng metric lẫn đọc thủ công. Không chạy lại generation với config khác.

In [ ]:
# Chọn 1 mẫu đã có trong locked test comparison.
# HINT: ưu tiên một dòng LoRA cải thiện rõ và một failure case.

"""Your code here"""

In [ ]:
# Tạo bảng 4 cột: dialect / baseline / LoRA / standard.
# HINT: dùng normalized_baseline và normalized_lora trong comparison.

"""Your code here"""

In [ ]:
# In CER/chrF của baseline và LoRA; viết 1-2 câu về bảo toàn nghĩa/task format.

"""Your code here"""

## Bài tập mở

1. Thay đổi LoRA rank (4, 8, 16), chọn bằng dev CER và chỉ chạy test cho cấu hình đã khóa.
2. Thử target modules khác (`k_proj`, `o_proj`) và giải thích vì sao.
3. So sánh decoding: `num_beams=1` (greedy) vs `num_beams=4` (beam search).
4. Tính correlation giữa CER và NLL recovery; giải thích vì sao correlation không
   biến NLL thành quality metric.
5. So sánh downstream recovery trên SENT và NLI.
6. Viết kết luận: normalization có giảm gap phương ngữ không? Fine-tune có giúp
   thêm không? Hạn chế và hướng tiếp theo.

In [ ]:
# STUDENT TASK 3 (EXTENSION) — Một thí nghiệm mở do nhóm thiết kế.
RUN_STUDENT_EXPERIMENT = False

def student_normalization_experiment(dev_baseline_df, nll_df):
    # HINT 1: chọn một biến chưa thử (rank, target module hoặc decoding).
    # HINT 2: chọn thiết kế bằng dev, không nhìn test.
    # HINT 3: trả về dict có "summary" (DataFrame) và "figure" (matplotlib Figure).
    """Your code here"""
    return None

if RUN_STUDENT_EXPERIMENT:
    result = student_normalization_experiment(dev_baseline, nll_frame)
    if result is None:
        raise NotImplementedError("Hoàn thành student_normalization_experiment trước khi bật cờ")
    assert isinstance(result, dict)
    assert {"summary", "figure"}.issubset(result)
    display(result["summary"])
    plt.show()

In [ ]:
# Lưu artifact nộp bài.
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
if not dev_baseline.empty:
    dev_baseline.to_csv(output_dir / "team_dev_baseline.csv", index=False)
    print("Saved", output_dir / "team_dev_baseline.csv")
if not nll_frame.empty:
    nll_frame.to_csv(output_dir / "team_dev_nll_diagnostic.csv", index=False)
    print("Saved", output_dir / "team_dev_nll_diagnostic.csv")